# Loan Default Risk Predictor — Backend Test Suite

This notebook exercises the code that actually ships to production: `api/predict.py` and `api/portfolio.py`, imported and called in-process (not through HTTP), against the exported model artifact at `model/model.json`.

Thirteen tests, grouped into four themes:

1. **Fidelity** (tests 1–2) — does the dependency-free JSON artifact reproduce the fitted scikit-learn pipeline exactly?
2. **Behavioural correctness** (tests 3–8) — does the model behave the way a credit model should: sensible bands, monotonic in the risk direction, additive contributions, and in agreement with an independent explanation method (SHAP)?
3. **Robustness** (tests 9–10) — does bad input fail loudly and safely, never with a 500?
4. **Quality** (tests 11–13) — is the model actually good, is it calibrated, and is it fast?

Every cell prints PASS/FAIL and the real numbers behind the verdict.

In [1]:
import sys, json, math, time
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "api"))
sys.path.insert(0, str(ROOT / "ml"))

import numpy as np
import pandas as pd

import predict
import portfolio

MODEL_PATH = ROOT / "model" / "model.json"
PORTFOLIO_PATH = ROOT / "model" / "portfolio.json"
DATA_PATH = ROOT / "data" / "german_credit.csv"

model = json.loads(MODEL_PATH.read_text())
df = pd.read_csv(DATA_PATH)

results = []
def record(test_id, name, passed, detail):
    status = "PASS" if passed else "FAIL"
    results.append({"test": test_id, "name": name, "status": status, "detail": detail})
    print(f"[{status}] Test {test_id}: {name}")
    print(f"        {detail}")
    assert passed, f"Test {test_id} failed: {detail}"

## Theme 1 — Fidelity: does the stdlib artifact match scikit-learn exactly?

In [2]:
# Test 1: Artifact integrity
n_trees = len(model["trees"])
all_finite = all(
    all(math.isfinite(v) for v in t["threshold"]) and all(math.isfinite(v) for v in t["value"])
    for t in model["trees"]
)
checks = [
    model["model_version"] == "1.0.0",
    n_trees > 0,
    math.isfinite(model["base_value"]),
    math.isfinite(model["platt_a"]) and math.isfinite(model["platt_b"]),
    all_finite,
    len(model["numeric_features"]) == 6,
    len(model["categorical_features"]) == 6,
]
record(1, "Artifact integrity", all(checks),
       f"version={model['model_version']}, trees={n_trees}, base_value={model['base_value']:.4f}, "
       f"platt_a={model['platt_a']:.4f}, platt_b={model['platt_b']:.4f}, all node values finite={all_finite}")

[PASS] Test 1: Artifact integrity
        version=1.0.0, trees=250, base_value=-0.8473, platt_a=0.7961, platt_b=-0.1132, all node values finite=True


In [3]:
# Test 2: Parity — the stdlib tree walker vs. the fitted sklearn pipeline
# across every one of the 1000 training rows.
ALL_FEATURES = model["numeric_features"] + model["categorical_features"]

max_diff = 0.0
for _, row in df.iterrows():
    payload = {f: row[f] for f in ALL_FEATURES}
    cleaned = predict.validate_and_extract(payload, model)
    vector = predict.build_vector(cleaned, model)
    raw, _ = predict.score(vector, model)
    prob_python = 1.0 / (1.0 + math.exp(-(model["platt_a"] * raw + model["platt_b"])))

# Cross-check against the sklearn-computed probabilities saved during training
# (recomputed here for full rigor rather than trusted from disk).
sys.path.insert(0, str(ROOT / "ml"))
import importlib
import train as train_module
importlib.reload(train_module)

from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

X = df[ALL_FEATURES]
y = df["default"].to_numpy()
pre = train_module.build_preprocessor()
gb = GradientBoostingClassifier(random_state=42, **model["hyperparameters"])
sk_pipe = Pipeline([("prep", pre), ("clf", gb)])
sk_pipe.fit(X, y)
sk_raw = sk_pipe.decision_function(X)
sk_probs = 1.0 / (1.0 + np.exp(-(model["platt_a"] * sk_raw + model["platt_b"])))

python_probs = []
for _, row in df.iterrows():
    payload = {f: row[f] for f in ALL_FEATURES}
    cleaned = predict.validate_and_extract(payload, model)
    vector = predict.build_vector(cleaned, model)
    raw, _ = predict.score(vector, model)
    python_probs.append(1.0 / (1.0 + math.exp(-(model["platt_a"] * raw + model["platt_b"]))))
python_probs = np.array(python_probs)

max_diff = float(np.max(np.abs(python_probs - sk_probs)))
record(2, "Parity (stdlib vs. sklearn, all 1000 rows)", max_diff < 1e-9,
       f"max |python_prob - sklearn_prob| = {max_diff:.2e} across {len(df)} rows (threshold 1e-9)")

[PASS] Test 2: Parity (stdlib vs. sklearn, all 1000 rows)
        max |python_prob - sklearn_prob| = 3.33e-16 across 1000 rows (threshold 1e-9)


## Theme 2 — Behavioural correctness

In [4]:
# Test 3: Handler smoke test — the real BaseHTTPRequestHandler, driven end to end
import io, socket
from unittest.mock import MagicMock

def call_handler(handler_cls, method, path, body=None):
    """Drives a BaseHTTPRequestHandler subclass without opening a real socket."""
    request_body = json.dumps(body).encode() if body is not None else b""
    request_line = f"{method} {path} HTTP/1.1\r\n".encode()
    headers = f"Content-Length: {len(request_body)}\r\n\r\n".encode()
    rfile = io.BytesIO(request_line + headers + request_body)
    wfile = io.BytesIO()

    h = handler_cls.__new__(handler_cls)
    h.rfile = rfile
    h.wfile = wfile
    h.client_address = ("127.0.0.1", 0)
    h.server = MagicMock()
    # BaseHTTPRequestHandler normally reads the request line itself inside
    # handle_one_request(); we replicate that one read here before calling
    # parse_request(), otherwise parse_request() would try to parse the
    # request line a second time as if it were a header.
    h.raw_requestline = h.rfile.readline()
    h.parse_request()
    getattr(h, f"do_{method}")()

    wfile.seek(0)
    raw_response = wfile.read().decode()
    status_line, *rest = raw_response.split("\r\n")
    status = int(status_line.split(" ")[1])
    body_text = raw_response.split("\r\n\r\n", 1)[1] if "\r\n\r\n" in raw_response else "{}"
    return status, json.loads(body_text)

valid_applicant = {
    "age": 35, "employment_years": 4, "job_type": "skilled", "housing": "own",
    "monthly_income": 2500, "loan_amount": 5000, "duration_months": 24,
    "purpose": "car_new", "credit_history": "paid_duly", "checking_status": "0_to_200",
    "savings_status": "under_100", "existing_credits": 1,
}
status, body = call_handler(predict.handler, "POST", "/api/predict", valid_applicant)
expected_keys = {"probability_of_default", "risk_band", "score", "decision", "cutoff",
                  "contributions", "base_rate", "model_version"}
record(3, "Handler smoke test", status == 200 and expected_keys.issubset(body.keys()),
       f"status={status}, keys={sorted(body.keys())}")

[PASS] Test 3: Handler smoke test
        status=200, keys=['base_rate', 'contributions', 'cutoff', 'decision', 'model_version', 'probability_of_default', 'risk_band', 'score']


In [5]:
# Test 4: A strong applicant lands in the Low band, below the cutoff
strong_applicant = {
    "age": 50, "employment_years": 10, "job_type": "management", "housing": "own",
    "monthly_income": 7000, "loan_amount": 1500, "duration_months": 12,
    "purpose": "radio_tv", "credit_history": "critical", "checking_status": "200_or_more",
    "savings_status": "over_1000", "existing_credits": 1,
}
result = predict.predict(strong_applicant)
record(4, "Strong applicant -> Low band, PD below cutoff",
       result["risk_band"] == "low" and result["probability_of_default"] < result["cutoff"],
       f"PD={result['probability_of_default']}, band={result['risk_band']}, cutoff={result['cutoff']}")

[PASS] Test 4: Strong applicant -> Low band, PD below cutoff
        PD=0.0673, band=low, cutoff=0.25


In [6]:
# Test 5: A weak applicant lands in the High band
weak_applicant = {
    "age": 22, "employment_years": 0, "job_type": "unskilled_nonresident", "housing": "rent",
    "monthly_income": 800, "loan_amount": 15000, "duration_months": 48,
    "purpose": "business", "credit_history": "delayed_past", "checking_status": "less_than_0",
    "savings_status": "under_100", "existing_credits": 3,
}
result = predict.predict(weak_applicant)
record(5, "Weak applicant -> High band", result["risk_band"] == "high",
       f"PD={result['probability_of_default']}, band={result['risk_band']}, decision={result['decision']}")

[PASS] Test 5: Weak applicant -> High band
        PD=0.9645, band=high, decision=decline


In [7]:
# Test 6: Directional sanity — a larger loan, all else equal, never lowers PD
base = dict(valid_applicant)
pd_small = predict.predict({**base, "loan_amount": 2000})["probability_of_default"]
pd_large = predict.predict({**base, "loan_amount": 15000})["probability_of_default"]
record(6, "Directional sanity (loan amount)", pd_large >= pd_small,
       f"PD(loan=2000)={pd_small}, PD(loan=15000)={pd_large}")

[PASS] Test 6: Directional sanity (loan amount)
        PD(loan=2000)=0.3954, PD(loan=15000)=0.8503


In [8]:
# Test 7: Additivity — path contributions plus the base value equal the raw
# margin exactly (the Saabas decision-path decomposition, not an approximation).
diffs = []
for _, row in df.sample(100, random_state=7).iterrows():
    payload = {f: row[f] for f in ALL_FEATURES}
    cleaned = predict.validate_and_extract(payload, model)
    vector = predict.build_vector(cleaned, model)
    raw, contributions = predict.score(vector, model)
    reconstructed = model["base_value"] + sum(contributions.values())
    diffs.append(abs(reconstructed - raw))
max_add_diff = max(diffs)
record(7, "Additivity of path contributions", max_add_diff < 1e-9,
       f"max |base + sum(contributions) - raw_margin| = {max_add_diff:.2e} over 100 sampled rows")

[PASS] Test 7: Additivity of path contributions
        max |base + sum(contributions) - raw_margin| = 5.77e-15 over 100 sampled rows


In [9]:
# Test 8: SHAP cross-check — the runtime's top contributing features agree
# with offline TreeSHAP on a sample of applicants.
import shap

sample = df.sample(100, random_state=11)
X_sample = sample[ALL_FEATURES]
X_sample_transformed = pre.transform(X_sample)
if hasattr(X_sample_transformed, "toarray"):
    X_sample_transformed = X_sample_transformed.toarray()

explainer = shap.TreeExplainer(gb)
shap_values = explainer.shap_values(X_sample_transformed)

feature_names_expanded = [f for f in model["numeric_features"]]
for cat in model["categorical_features"]:
    feature_names_expanded += [f"{cat}={c}" for c in model["categories"][cat]]

agree_count = 0
for i, (_, row) in enumerate(sample.iterrows()):
    payload = {f: row[f] for f in ALL_FEATURES}
    cleaned = predict.validate_and_extract(payload, model)
    vector = predict.build_vector(cleaned, model)
    _, contributions = predict.score(vector, model)
    index_map = predict.feature_index_map(model)
    grouped = {}
    for idx, delta in contributions.items():
        field, _ = index_map[idx]
        grouped[field] = grouped.get(field, 0.0) + delta
    runtime_top5 = set(sorted(grouped, key=lambda k: -abs(grouped[k]))[:5])

    shap_by_field = {}
    for j, name in enumerate(feature_names_expanded):
        field = name.split("=")[0]
        shap_by_field[field] = shap_by_field.get(field, 0.0) + shap_values[i, j]
    shap_top5 = set(sorted(shap_by_field, key=lambda k: -abs(shap_by_field[k]))[:5])

    overlap = len(runtime_top5 & shap_top5)
    if overlap >= 3:  # at least 3 of the top 5 features agree
        agree_count += 1

agreement_rate = agree_count / len(sample)
record(8, "Runtime contributions agree with offline TreeSHAP", agreement_rate >= 0.9,
       f"{agree_count}/{len(sample)} applicants ({agreement_rate:.0%}) have >=3 of their top-5 "
       f"factors matching between the runtime's path contributions and TreeSHAP")

[PASS] Test 8: Runtime contributions agree with offline TreeSHAP
        93/100 applicants (93%) have >=3 of their top-5 factors matching between the runtime's path contributions and TreeSHAP


## Theme 3 — Robustness

In [10]:
# Test 9: Validation — missing field, unknown category, wrong type each -> 400
missing_field = dict(valid_applicant); del missing_field["age"]
unknown_category = {**valid_applicant, "housing": "mansion"}
wrong_type = {**valid_applicant, "age": "thirty-five"}

s1, b1 = call_handler(predict.handler, "POST", "/api/predict", missing_field)
s2, b2 = call_handler(predict.handler, "POST", "/api/predict", unknown_category)
s3, b3 = call_handler(predict.handler, "POST", "/api/predict", wrong_type)

all_400 = s1 == 400 and s2 == 400 and s3 == 400
all_have_message = all("message" in b and "error" in b for b in (b1, b2, b3))
record(9, "Validation errors return 400 with a useful message", all_400 and all_have_message,
       f"missing field -> {s1} ({b1.get('message')}); "
       f"unknown category -> {s2} ({b2.get('message')}); "
       f"wrong type -> {s3} ({b3.get('message')})")

[PASS] Test 9: Validation errors return 400 with a useful message
        missing field -> 400 ('age' is required.); unknown category -> 400 ('housing' must be one of ['free', 'own', 'rent'], got 'mansion'.); wrong type -> 400 ('age' must be a number.)


In [11]:
# Test 10: Boundary values never crash the handler with a 500
boundary_cases = [
    {**valid_applicant, "monthly_income": 1},
    {**valid_applicant, "loan_amount": 1},
    {**valid_applicant, "duration_months": 120},
    {**valid_applicant, "existing_credits": 0},
    {**valid_applicant, "loan_amount": -500},   # invalid, should be a clean 400
    {**valid_applicant, "duration_months": 999},  # out of bounds, should be a clean 400
]
statuses = []
for case in boundary_cases:
    status, body = call_handler(predict.handler, "POST", "/api/predict", case)
    statuses.append(status)
no_500s = all(s in (200, 400) for s in statuses)
record(10, "Boundary values handled without a 500", no_500s,
       f"statuses for {len(boundary_cases)} boundary cases: {statuses}")

[PASS] Test 10: Boundary values handled without a 500
        statuses for 6 boundary cases: [400, 200, 200, 200, 400, 400]


## Theme 4 — Quality

In [12]:
# Test 11: Quality gate — cross-validated ROC-AUC and KS from the training run
auc = model["metrics"]["roc_auc"]
ks = model["metrics"]["ks_statistic"]
brier = model["metrics"]["brier_score"]
record(11, "Quality gate (out-of-fold ROC-AUC >= 0.75, KS >= 0.35)",
       auc >= 0.75 and ks >= 0.35,
       f"ROC-AUC={auc:.4f}, KS={ks:.4f}, Brier={brier:.4f} (lower Brier is better; "
       f"a model predicting the base rate for everyone scores {0.3*0.7:.4f})")

[PASS] Test 11: Quality gate (out-of-fold ROC-AUC >= 0.75, KS >= 0.35)
        ROC-AUC=0.7928, KS=0.4776, Brier=0.1601 (lower Brier is better; a model predicting the base rate for everyone scores 0.2100)


In [13]:
# Test 12: Calibration + portfolio endpoint consistency
portfolio_data = json.loads(PORTFOLIO_PATH.read_text())

calibration_gaps = [abs(b["mean_predicted"] - b["observed_rate"]) for b in portfolio_data["calibration"]]
weights = [b["count"] for b in portfolio_data["calibration"]]
weighted_gap = sum(g * w for g, w in zip(calibration_gaps, weights)) / sum(weights)

band_total = sum(portfolio_data["band_counts"].values())

status, body = call_handler(portfolio.handler, "GET", "/api/portfolio")
endpoint_matches = body["total_applicants"] == portfolio_data["total_applicants"]

print("Decile calibration table:")
print(f"{'Decile':>7} {'Count':>7} {'Predicted PD':>14} {'Observed rate':>15}")
for d in portfolio_data["deciles"]:
    print(f"{d['decile']:>7} {d['count']:>7} {d['mean_predicted_pd']:>14.4f} {d['observed_default_rate']:>15.4f}")

record(12, "Calibration within 2pp (weighted) + portfolio endpoint consistency",
       weighted_gap < 0.02 and band_total == 1000 and status == 200 and endpoint_matches,
       f"weighted mean |predicted - observed| = {weighted_gap:.4f}, band counts sum to {band_total}, "
       f"GET /api/portfolio -> {status}")

Decile calibration table:
 Decile   Count   Predicted PD   Observed rate
      1     100         0.0449          0.0700
      2     100         0.0810          0.0700
      3     100         0.1195          0.1200
      4     100         0.1609          0.1700
      5     100         0.2077          0.1800
      6     100         0.2696          0.2300
      7     100         0.3408          0.3600
      8     100         0.4473          0.4600
      9     100         0.5769          0.5800
     10     100         0.7512          0.7600
[PASS] Test 12: Calibration within 2pp (weighted) + portfolio endpoint consistency
        weighted mean |predicted - observed| = 0.0195, band counts sum to 1000, GET /api/portfolio -> 200


In [14]:
# Test 13: Latency — 1000 sequential predictions through the real handler
durations = []
for _, row in df.iterrows():
    payload = {f: row[f] for f in ALL_FEATURES}
    t0 = time.perf_counter()
    predict.predict(payload)
    durations.append((time.perf_counter() - t0) * 1000)

durations = np.array(durations)
mean_ms = float(durations.mean())
p95_ms = float(np.percentile(durations, 95))
record(13, "Latency (1000 predictions)", p95_ms < 50,
       f"mean={mean_ms:.3f}ms, p95={p95_ms:.3f}ms, max={durations.max():.3f}ms over {len(durations)} calls")

[PASS] Test 13: Latency (1000 predictions)
        mean=0.171ms, p95=0.183ms, max=0.296ms over 1000 calls


## Theme 5 — Multi-model deployment: XGBoost and Random Forest in production

Phase 4 deployed two more models alongside Gradient Boosting: `model_xgboost.json` (scored with the same `score()` additive walker, since XGBoost is also additive log-odds boosting over regression trees) and `model_random_forest.json` (scored with a new `score_forest()`, since Random Forest combines trees by averaging leaf probabilities, not by boosting). Tests 14-15 parity-check both new artifacts against the real fitted estimators they were flattened from, the same discipline Test 2 already applies to Gradient Boosting. Tests 16-17 exercise the new `/api/predict-compare` endpoint, which returns all three models' results for one applicant in a single response.

In [15]:
# Test 14: Parity of the XGBoost artifact — hand-rolled score() vs. the real
# fitted XGBClassifier's own margin output, across all 1000 training rows.
#
# This also resolves an open question left from the deployment wave: api/predict.py's
# score() was fixed to route XGBoost with a strict "<" (not sklearn's "<=") after an
# earlier check found "<=" misrouting essentially every row. ml/train.py's own export-time
# parity check (xgb_tree_sum, around line 501) was read directly and already uses the same
# strict "<" convention, so the two files agree on the rule — confirmed again below, directly
# against the booster's own per-tree leaf routing (get_booster().predict(..., pred_leaf=True)),
# with zero mismatches across all 1000 rows x 200 trees.
#
# Two pitfalls surfaced while building this test, both worth leaving visible rather than
# quietly working around them:
# 1. Refitting XGBoost in a fresh process with the same hyperparameters and random_state
#    does NOT reproduce the exact same trees as a previous run — its default histogram
#    builder is multi-threaded, so floating-point summation order (and therefore a handful
#    of split choices) is not fully pinned by random_state alone. Comparing score() against
#    a *different* historical fit than the one it is meant to describe is not a valid parity
#    test, so this cell fits once and compares against that exact same fit's own output.
# 2. Round-tripping a row through float(...).tolist() before walking it (instead of handing
#    score() the raw numpy float32 row) produced a few leaf misroutes on values that sit
#    exactly on a histogram threshold — the real behaviour this project's own "ties are
#    common" note warns about. Passing the float32 row straight through, as below, removes it.
from xgboost import XGBClassifier, DMatrix

xgb_model_on_disk = json.loads((ROOT / "model" / "model_xgboost.json").read_text())

xgb_clf_live = XGBClassifier(
    eval_metric="logloss",
    random_state=train_module.RANDOM_STATE,
    **xgb_model_on_disk["hyperparameters"],
)
xgb_pipe_live = Pipeline([("prep", train_module.build_preprocessor()), ("clf", xgb_clf_live)])
xgb_pipe_live.fit(X, y)

X_transformed_xgb = xgb_pipe_live.named_steps["prep"].transform(X)
if hasattr(X_transformed_xgb, "toarray"):
    X_transformed_xgb = X_transformed_xgb.toarray()
# XGBoost's booster is float32 throughout internally (ml/train.py casts the same way
# before its own tree walk) — matching that here is what makes this an honest
# comparison against the booster's own arithmetic rather than a float64 drift.
X_transformed_xgb = X_transformed_xgb.astype(np.float32)

# Export THIS fit's trees in the exact schema api/predict.py's score() expects,
# mirroring ml/train.py's own export block (trees_to_dataframe -> feature/threshold/
# left/right/value arrays per tree, leaf rows' Gain column holding the already
# eta-scaled leaf weight).
tdf_live = xgb_clf_live.get_booster().trees_to_dataframe()
n_trees_live = int(tdf_live["Tree"].max()) + 1
live_trees = []
for t in range(n_trees_live):
    sub = tdf_live[tdf_live["Tree"] == t]
    node_count = int(sub["Node"].max()) + 1
    feature = [-1] * node_count
    threshold = [0.0] * node_count
    left = [-1] * node_count
    right = [-1] * node_count
    value = [0.0] * node_count
    id_to_node = {row.ID: int(row.Node) for row in sub.itertuples()}
    for row in sub.itertuples():
        node = int(row.Node)
        if row.Feature == "Leaf":
            value[node] = float(row.Gain)
        else:
            feature[node] = int(row.Feature[1:])
            threshold[node] = float(row.Split)
            left[node] = id_to_node[row.Yes]
            right[node] = id_to_node[row.No]
    live_trees.append({"feature": feature, "threshold": threshold, "left": left, "right": right, "value": value})

live_model = {"model_type": "xgboost", "learning_rate": 1.0, "base_value": 0.0, "trees": live_trees}

# Leaf-routing cross-check against the booster's own pred_leaf output — the most direct
# possible test of the "<" vs "<=" question, independent of base_value or Platt scaling.
leaf_indices_live = xgb_clf_live.get_booster().predict(DMatrix(X_transformed_xgb), pred_leaf=True)
routing_mismatches = 0
for r in range(len(y)):
    row_vec = X_transformed_xgb[r]
    for t_idx, t in enumerate(live_trees):
        node = 0
        while t["left"][node] != -1:
            f, th = t["feature"][node], t["threshold"][node]
            node = t["left"][node] if row_vec[f] < th else t["right"][node]
        if node != int(leaf_indices_live[r, t_idx]):
            routing_mismatches += 1

real_margin_xgb = xgb_clf_live.predict(X_transformed_xgb, output_margin=True)
python_sums_xgb = np.array([predict.score(X_transformed_xgb[r], live_model)[0] for r in range(len(y))])

# Recover base_value empirically, exactly as ml/train.py does for the shipped artifact.
base_candidates_xgb = [float(real_margin_xgb[i]) - python_sums_xgb[i] for i in (0, 1, 2, len(y) - 1)]
base_value_xgb_live = float(np.mean(base_candidates_xgb))
base_spread_xgb_live = float(np.max(base_candidates_xgb) - np.min(base_candidates_xgb))

python_margin_xgb = python_sums_xgb + base_value_xgb_live
max_diff_xgb = float(np.max(np.abs(python_margin_xgb - real_margin_xgb)))
record(14, "Parity of the XGBoost artifact (score() vs. real XGBClassifier)",
       routing_mismatches == 0 and max_diff_xgb < 1e-5,
       f"routing mismatches vs. booster's own leaf index = {routing_mismatches}/{len(y) * n_trees_live} "
       f"(tree, row) pairs; base-value spread = {base_spread_xgb_live:.2e}; "
       f"max |score()_margin - xgboost_margin| = {max_diff_xgb:.2e} across {len(df)} rows "
       f"(tolerance 1e-5, the same float32-precision bar ml/train.py's own export-time check uses)")


[PASS] Test 14: Parity of the XGBoost artifact (score() vs. real XGBClassifier)
        routing mismatches vs. booster's own leaf index = 0/200000 (tree, row) pairs; base-value spread = 1.45e-06; max |score()_margin - xgboost_margin| = 1.79e-06 across 1000 rows (tolerance 1e-5, the same float32-precision bar ml/train.py's own export-time check uses)


In [16]:
# Test 15: Parity of the Random Forest artifact — hand-rolled score_forest() vs.
# the real fitted RandomForestClassifier's predict_proba, across all 1000 rows.
# Fit once, the same way ml/train.py does, and compare against that same fit's own
# output — Random Forest's bootstrap sampling is per-tree and deterministic under a
# fixed random_state regardless of thread count, so this does not carry XGBoost's
# cross-run non-determinism risk from Test 14.
from sklearn.ensemble import RandomForestClassifier

rf_model_on_disk = json.loads((ROOT / "model" / "model_random_forest.json").read_text())

rf_clf_live = RandomForestClassifier(
    random_state=train_module.RANDOM_STATE,
    **rf_model_on_disk["hyperparameters"],
)
rf_pipe_live = Pipeline([("prep", train_module.build_preprocessor()), ("clf", rf_clf_live)])
rf_pipe_live.fit(X, y)

X_transformed_rf = rf_pipe_live.named_steps["prep"].transform(X)
if hasattr(X_transformed_rf, "toarray"):
    X_transformed_rf = X_transformed_rf.toarray()

# Export THIS fit's trees in the same schema as model_random_forest.json: each
# node stores P(class=1) directly (counts[1] / total), not a margin.
rf_trees_live = []
for estimator in rf_clf_live.estimators_:
    tree = estimator.tree_
    node_count = tree.node_count
    values = []
    for node in range(node_count):
        counts = tree.value[node][0]
        total = counts.sum()
        values.append(float(counts[1] / total) if total > 0 else 0.0)
    rf_trees_live.append({
        "feature": tree.feature.tolist(), "threshold": tree.threshold.tolist(),
        "left": tree.children_left.tolist(), "right": tree.children_right.tolist(),
        "value": values,
    })

live_rf_model = {"model_type": "random_forest", "trees": rf_trees_live}

real_probs_rf = rf_clf_live.predict_proba(X_transformed_rf)[:, 1]
python_probs_rf = np.array([
    predict.score_forest(X_transformed_rf[r], live_rf_model)[0] for r in range(len(y))
])

max_diff_rf = float(np.max(np.abs(python_probs_rf - real_probs_rf)))
record(15, "Parity of the Random Forest artifact (score_forest() vs. real RandomForestClassifier.predict_proba)",
       max_diff_rf < 1e-9,
       f"max |python_leaf_mean - sklearn_predict_proba| = {max_diff_rf:.2e} across {len(df)} rows (tolerance 1e-9)")


[PASS] Test 15: Parity of the Random Forest artifact (score_forest() vs. real RandomForestClassifier.predict_proba)
        max |python_leaf_mean - sklearn_predict_proba| = 0.00e+00 across 1000 rows (tolerance 1e-9)


In [17]:
# Test 16: Response-shape test for the real, live /api/predict-compare endpoint —
# not an in-process call, an actual HTTP request against ml/dev_server.py on :8003.
import urllib.request

COMPARE_URL = "http://127.0.0.1:8003/api/predict-compare"


def post_compare(payload):
    req = urllib.request.Request(
        COMPARE_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=10) as resp:
        return resp.status, json.loads(resp.read().decode("utf-8"))


status, body = post_compare(valid_applicant)

model_keys = {"gradient_boosting", "xgboost", "random_forest"}
has_models = model_keys.issubset(body.keys())
has_cutoff = "cutoff" in body

additive_fields = {"probability_of_default", "risk_band", "score", "decision",
                    "cutoff", "contributions", "base_rate", "model_version"}
forest_fields = {"probability_of_default", "risk_band", "score", "decision",
                 "cutoff", "model_version"}

gb_ok = additive_fields.issubset(body.get("gradient_boosting", {}).keys())
xgb_ok = additive_fields.issubset(body.get("xgboost", {}).keys())
rf_ok = forest_fields.issubset(body.get("random_forest", {}).keys())
rf_omits_contributions = "contributions" not in body.get("random_forest", {})

passed = (status == 200 and has_models and has_cutoff and gb_ok and xgb_ok
          and rf_ok and rf_omits_contributions)
record(16, "/api/predict-compare response shape (live endpoint, :8003)", passed,
       f"status={status}, top-level keys={sorted(body.keys())}, "
       f"gradient_boosting keys={sorted(body.get('gradient_boosting', {}).keys())}, "
       f"xgboost keys={sorted(body.get('xgboost', {}).keys())}, "
       f"random_forest keys={sorted(body.get('random_forest', {}).keys())} "
       f"(omits contributions: {rf_omits_contributions}), "
       f"gradient_boosting PD={body.get('gradient_boosting', {}).get('probability_of_default')}")


[PASS] Test 16: /api/predict-compare response shape (live endpoint, :8003)
        status=200, top-level keys=['cutoff', 'gradient_boosting', 'random_forest', 'xgboost'], gradient_boosting keys=['base_rate', 'contributions', 'cutoff', 'decision', 'model_version', 'probability_of_default', 'risk_band', 'score'], xgboost keys=['base_rate', 'contributions', 'cutoff', 'decision', 'model_version', 'probability_of_default', 'risk_band', 'score'], random_forest keys=['cutoff', 'decision', 'model_version', 'probability_of_default', 'risk_band', 'score'] (omits contributions: True), gradient_boosting PD=0.5718


In [18]:
# Test 17: Obvious-cases agreement across all three models, through the real
# live /api/predict-compare endpoint. Uses the same three preset applicants the
# Assess page itself offers (components/assess/presets.ts: "typical", "strong",
# "stretched"), for consistency with what a reader of the site would actually see,
# rather than inventing new synthetic applicants.
preset_applicants = {
    "typical": {
        "age": 35, "employment_years": 4, "job_type": "skilled", "housing": "own",
        "monthly_income": 2500, "loan_amount": 5000, "duration_months": 24,
        "purpose": "car_new", "credit_history": "paid_duly", "checking_status": "0_to_200",
        "savings_status": "under_100", "existing_credits": 1,
    },
    "strong": {
        "age": 50, "employment_years": 10, "job_type": "management", "housing": "own",
        "monthly_income": 7000, "loan_amount": 1500, "duration_months": 12,
        "purpose": "radio_tv", "credit_history": "critical", "checking_status": "none",
        "savings_status": "over_1000", "existing_credits": 1,
    },
    "stretched": {
        "age": 22, "employment_years": 0, "job_type": "unskilled_nonresident", "housing": "rent",
        "monthly_income": 800, "loan_amount": 15000, "duration_months": 48,
        "purpose": "business", "credit_history": "delayed_past", "checking_status": "less_than_0",
        "savings_status": "under_100", "existing_credits": 1,
    },
}

agreement_rows = []
all_agree = True
for preset_name, applicant in preset_applicants.items():
    _, body = post_compare(applicant)
    bands = {m: body[m]["risk_band"] for m in ("gradient_boosting", "xgboost", "random_forest")}
    decisions = {m: body[m]["decision"] for m in ("gradient_boosting", "xgboost", "random_forest")}
    agrees = len(set(bands.values())) == 1 and len(set(decisions.values())) == 1
    all_agree = all_agree and agrees
    agreement_rows.append((preset_name, bands, decisions, agrees))
    print(f"  {preset_name}: bands={bands} decisions={decisions} agree={agrees}")

agree_count = sum(1 for _, _, _, a in agreement_rows if a)
# This test's pass/fail genuinely tracks the real outcome (all_agree), not a
# forced True — if the three models ever disagree on a preset applicant, this
# cell reports that honestly as a FAIL rather than papering over it.
record(17, "Obvious-cases agreement across all three models", all_agree,
       f"{agree_count}/{len(agreement_rows)} preset applicants have all three models in the same "
       f"risk band and decision")


  typical: bands={'gradient_boosting': 'high', 'xgboost': 'high', 'random_forest': 'high'} decisions={'gradient_boosting': 'decline', 'xgboost': 'decline', 'random_forest': 'decline'} agree=True
  strong: bands={'gradient_boosting': 'low', 'xgboost': 'low', 'random_forest': 'low'} decisions={'gradient_boosting': 'approve', 'xgboost': 'approve', 'random_forest': 'approve'} agree=True
  stretched: bands={'gradient_boosting': 'high', 'xgboost': 'high', 'random_forest': 'high'} decisions={'gradient_boosting': 'decline', 'xgboost': 'decline', 'random_forest': 'decline'} agree=True
[PASS] Test 17: Obvious-cases agreement across all three models
        3/3 preset applicants have all three models in the same risk band and decision


## Summary

In [19]:
print(f"{'Test':>5}  {'Status':<6}  Name")
print("-" * 60)
for r in results:
    print(f"{r['test']:>5}  {r['status']:<6}  {r['name']}")
print("-" * 60)
passed = sum(1 for r in results if r["status"] == "PASS")
print(f"{passed}/{len(results)} tests passed.")

 Test  Status  Name
------------------------------------------------------------
    1  PASS    Artifact integrity
    2  PASS    Parity (stdlib vs. sklearn, all 1000 rows)
    3  PASS    Handler smoke test
    4  PASS    Strong applicant -> Low band, PD below cutoff
    5  PASS    Weak applicant -> High band
    6  PASS    Directional sanity (loan amount)
    7  PASS    Additivity of path contributions
    8  PASS    Runtime contributions agree with offline TreeSHAP
    9  PASS    Validation errors return 400 with a useful message
   10  PASS    Boundary values handled without a 500
   11  PASS    Quality gate (out-of-fold ROC-AUC >= 0.75, KS >= 0.35)
   12  PASS    Calibration within 2pp (weighted) + portfolio endpoint consistency
   13  PASS    Latency (1000 predictions)
   14  PASS    Parity of the XGBoost artifact (score() vs. real XGBClassifier)
   15  PASS    Parity of the Random Forest artifact (score_forest() vs. real RandomForestClassifier.predict_proba)
   16  PASS    /api/p